In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier

In [4]:
df = pd.read_csv("data/data.csv")

X = df.drop(columns=["verification.result"])
y = df["verification.result"].astype(int)

X["process.total_capacity"] = (
    X["process.b1.capacity"] + X["process.b2.capacity"] + 
    X["process.b3.capacity"] + X["process.b4.capacity"]
)

X["price_per_capacity"] = X["property.price"] / (X["process.total_capacity"] + 1)

X = pd.get_dummies(X, drop_first=True)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

base_model = LogisticRegression(max_iter=1000)
base_model.fit(X_train, y_train)
print("Base model")
print(classification_report(y_test, base_model.predict(X_test)))

bagging = RandomForestClassifier(random_state=42)
bagging.fit(X_train, y_train)
print("Bagging")
print(classification_report(y_test, bagging.predict(X_test)))

boosting = GradientBoostingClassifier(random_state=42)
boosting.fit(X_train, y_train)
print("Boosting")
print(classification_report(y_test, boosting.predict(X_test)))

stacking = StackingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(random_state=42)),
        ("gb", GradientBoostingClassifier(random_state=42))
    ],
    final_estimator=LogisticRegression(max_iter=1000)
)
stacking.fit(X_train, y_train)
print("Stacking")
print(classification_report(y_test, stacking.predict(X_test)))

Base model
              precision    recall  f1-score   support

           0       0.90      0.99      0.94       267
           1       0.80      0.30      0.44        40

    accuracy                           0.90       307
   macro avg       0.85      0.64      0.69       307
weighted avg       0.89      0.90      0.88       307

Bagging
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       267
           1       1.00      0.95      0.97        40

    accuracy                           0.99       307
   macro avg       1.00      0.97      0.99       307
weighted avg       0.99      0.99      0.99       307

Boosting
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       267
           1       1.00      0.97      0.99        40

    accuracy                           1.00       307
   macro avg       1.00      0.99      0.99       307
weighted avg       1.00      1.00      1.00   